# 🎬 NLP Sentiment Analysis — IMDb Movie Reviews

**Project:** Binary text classification using machine learning  
**Dataset:** IMDb Large Movie Review Dataset (Maas et al., 2011)  
**Goal:** Classify movie reviews as *Positive* or *Negative*  

---

## Pipeline Overview

```
Raw Text → Preprocessing → TF-IDF Vectorisation → Model Training → Evaluation → Visualisation
```

**Models compared:**
- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM (LinearSVC)

**Metrics:** Accuracy, F1-Score, Confusion Matrix, Classification Report, Cross-Validation

## 0 · Setup & Imports

In [ ]:
import os, sys, re, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils import shuffle

warnings.filterwarnings('ignore')

# Aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

RANDOM_STATE = 42
TEST_SIZE    = 0.20
MAX_FEATURES = 50_000

print('✓ All libraries imported successfully')

## 1 · Load Data

In [ ]:
CSV_PATH = 'data/imdb_sample.csv'

if not os.path.exists(CSV_PATH):
    print('Dataset not found — generating synthetic version …')
    os.system('python generate_data.py --synthetic')

df = pd.read_csv(CSV_PATH)
df = shuffle(df, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
display(df.head(3))

In [ ]:
# ── Class distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
axes[0].bar(['Negative', 'Positive'], counts.values, color=['#e74c3c','#2ecc71'], alpha=0.85)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Reviews')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontsize=11)

# Review length distribution
df['length'] = df['text'].str.split().str.len()
axes[1].hist(df[df['label']==1]['length'], bins=40, alpha=0.65,
             color='#2ecc71', label='Positive')
axes[1].hist(df[df['label']==0]['length'], bins=40, alpha=0.65,
             color='#e74c3c', label='Negative')
axes[1].set_title('Review Length Distribution (words)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Avg review length: {df["length"].mean():.0f} words')

## 2 · Text Preprocessing

In [ ]:
def preprocess_text(texts):
    """
    Light-touch cleaning:
    1. Strip whitespace
    2. Collapse consecutive spaces
    3. Lower-case

    Heavier normalisation (stopwords, stemming, lemmatisation) is handled
    inside TfidfVectorizer so the full pipeline stays serialisable.
    """
    cleaned = []
    for t in texts:
        t = t.strip()
        t = re.sub(r'\s+', ' ', t)
        t = t.lower()
        cleaned.append(t)
    return cleaned

X_raw = df['text'].tolist()
y     = df['label'].values
X     = preprocess_text(X_raw)

# Before / After sample
print('BEFORE:', X_raw[0][:120])
print()
print('AFTER :', X[0][:120])

## 3 · Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')
print(f'Positive in train: {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'Positive in test : {y_test.sum():,}  ({y_test.mean()*100:.1f}%)')

## 4 · TF-IDF Vectorisation

We use `TfidfVectorizer` to convert raw text into a numeric feature matrix.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `ngram_range` | (1, 2) | Capture word pairs like "not good" |
| `max_features` | 50,000 | Balance vocab size vs memory |
| `sublinear_tf` | True | Dampen high-frequency term counts |
| `min_df` | 2 | Ignore extremely rare tokens |
| `stop_words` | english | Remove uninformative words |

In [ ]:
# Standalone demo — in practice the vectoriser lives inside each Pipeline
demo_vec = TfidfVectorizer(
    ngram_range=(1, 2), max_features=MAX_FEATURES,
    sublinear_tf=True,  min_df=2, stop_words='english'
)
X_train_tfidf = demo_vec.fit_transform(X_train)

print(f'Vocabulary size : {len(demo_vec.vocabulary_):,}')
print(f'Feature matrix  : {X_train_tfidf.shape}  (sparse)')
print(f'Memory (sparse) : {X_train_tfidf.data.nbytes / 1e6:.1f} MB')

# Peek at top features by mean TF-IDF score
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).flatten()
top_idx    = mean_tfidf.argsort()[-15:][::-1]
feat_names = demo_vec.get_feature_names_out()
print('\nTop 15 features by mean TF-IDF:')
for i in top_idx:
    print(f'  {feat_names[i]:30s}  {mean_tfidf[i]:.5f}')

## 5 · Build & Train Model Pipelines

In [ ]:
TFIDF_PARAMS = dict(
    ngram_range=(1, 2), max_features=MAX_FEATURES,
    sublinear_tf=True, min_df=2, stop_words='english'
)

pipelines = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   MultinomialNB(alpha=0.1)),
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LogisticRegression(C=5.0, max_iter=1000, solver='lbfgs',
                                     random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
    ]),
}

results = {}

for name, pipe in pipelines.items():
    print(f'Training {name} …', end=' ')
    t0      = time.time()
    pipe.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_pred = pipe.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average='weighted')
    cm     = confusion_matrix(y_test, y_pred)

    results[name] = {'accuracy': acc, 'f1': f1,
                     'confusion_matrix': cm, 'pipeline': pipe,
                     'train_time': elapsed, 'y_pred': y_pred}
    print(f'done ({elapsed:.1f}s)  →  Accuracy: {acc:.4f}  F1: {f1:.4f}')

## 6 · Evaluation Metrics

In [ ]:
for name, data in results.items():
    print(f'\n{'─'*55}')
    print(f'  {name}')
    print(f'{'─'*55}')
    print(classification_report(y_test, data['y_pred'],
                                target_names=['Negative','Positive']))

In [ ]:
summary = pd.DataFrame([
    {'Model': n,
     'Accuracy': f"{d['accuracy']*100:.2f}%",
     'F1-Score': f"{d['f1']:.4f}",
     'Train Time (s)': f"{d['train_time']:.2f}"}
    for n, d in results.items()
])
display(summary.style.set_caption('Model Performance Summary'))

## 7 · Visualisations

In [ ]:
# ── Model Comparison Bar Chart ───────────────────────────────────────────────
names      = list(results.keys())
accuracies = [results[n]['accuracy'] for n in names]
f1_scores  = [results[n]['f1']       for n in names]
PALETTE    = ['#4C72B0', '#DD8452', '#55A868']

x     = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, accuracies, width, label='Accuracy',  color='#4C72B0')
b2 = ax.bar(x + width/2, f1_scores,  width, label='F1-Score',  color='#DD8452')

for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0.80, 1.01)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Accuracy & F1-Score', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrices ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = ['Negative', 'Positive']

for ax, (name, data) in zip(axes, results.items()):
    sns.heatmap(data['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax,
                linewidths=0.5, linecolor='white',
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Top Features — Logistic Regression ──────────────────────────────────────
pipe       = results['Logistic Regression']['pipeline']
vectorizer = pipe.named_steps['tfidf']
clf        = pipe.named_steps['clf']
feat_names = np.array(vectorizer.get_feature_names_out())
coefs      = clf.coef_[0]
N          = 20

top_pos = np.argsort(coefs)[-N:][::-1]
top_neg = np.argsort(coefs)[:N]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, idx, title, colour in [
    (axes[0], top_pos, 'Top Positive Features', '#2ecc71'),
    (axes[1], top_neg, 'Top Negative Features', '#e74c3c'),
]:
    ax.barh(feat_names[idx][::-1], coefs[idx][::-1], color=colour, alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Coefficient Weight')
    ax.grid(axis='x', linestyle='--', alpha=0.5)

fig.suptitle(f'Logistic Regression — Top {N} Features by Sentiment Class',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8 · Cross-Validation

In [ ]:
print('5-Fold Cross-Validation Accuracy\n')
cv_results = {}
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5,
                             scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:25s}  {scores.mean():.4f} ± {scores.std():.4f}')

# Box-plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
           medianprops=dict(color='black', linewidth=2))
ax.set_ylabel('CV Accuracy')
ax.set_title('5-Fold Cross-Validation Results', fontsize=13, fontweight='bold')
ax.set_ylim(0.80, 1.00)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 9 · Live Prediction Demo

In [ ]:
best_name = max(results, key=lambda n: results[n]['accuracy'])
best_pipe = results[best_name]['pipeline']
LABEL_MAP = {0: 'Negative 😞', 1: 'Positive 😊'}

test_reviews = [
    'This movie was absolutely brilliant! The acting was superb.',
    'Terrible film. Complete waste of two hours. I want a refund.',
    'An average movie — some good moments, some boring ones.',
    'The cinematography was stunning but the plot left much to be desired.',
    'One of the greatest films ever made. A true masterpiece!',
]

print(f'Best model: {best_name}\n')
for review in test_reviews:
    pred  = best_pipe.predict([review])[0]
    label = LABEL_MAP[pred]
    print(f'  {label}  →  "{review[:70]}"')

## 10 · Conclusions

| Model | Accuracy | F1-Score |
|-------|----------|----------|
| Naive Bayes | ~88% | ~0.88 |
| Logistic Regression | ~90% | ~0.90 |
| Linear SVM | ~91% | ~0.91 |

**Key takeaways:**
- TF-IDF with bigrams is a powerful baseline for text classification
- LinearSVC achieves the best accuracy with fast training time
- Logistic Regression provides interpretable feature weights
- All models significantly outperform a random baseline (50%)

**Potential improvements:**
- Use pre-trained word embeddings (Word2Vec, GloVe, FastText)
- Fine-tune a transformer model (BERT, DistilBERT) for ~95%+ accuracy
- Hyperparameter search with GridSearchCV / RandomizedSearchCV
- Ensemble methods (Voting Classifier)